# FIGURAS DE MÉRITO

In [1]:
from scipy.ndimage import binary_erosion
from scipy.spatial import cKDTree
from scipy.spatial.distance import directed_hausdorff
import numpy as np

def get_boundary(mask, width=2):
    eroded = binary_erosion(mask, iterations=width)
    return mask & ~eroded

def boundary_iou(pred_mask, gt_mask, width=2):
    pred_boundary = get_boundary(pred_mask, width)
    gt_boundary   = get_boundary(gt_mask, width)
    intersection  = np.logical_and(pred_boundary, gt_boundary).sum()
    union         = np.logical_or(pred_boundary, gt_boundary).sum()
    return intersection / union if union > 0 else 0

# TARDA MUCHO PORQUE TIENE QUE COGER TODOS LOS PUNTOS DE LA MÁSCARA
"""def assd(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    
    d_pred_to_gt = cKDTree(gt_points).query(pred_points)[0].mean()
    d_gt_to_pred = cKDTree(pred_points).query(gt_points)[0].mean()
    
    return (d_pred_to_gt + d_gt_to_pred) / 2"""

def hausdorff_95(pred_mask, gt_mask):
    pred_points = np.argwhere(pred_mask)
    gt_points   = np.argwhere(gt_mask)
    if len(pred_points) == 0 or len(gt_points) == 0:
        return 0
    d1 = directed_hausdorff(pred_points, gt_points)[0]
    d2 = directed_hausdorff(gt_points, pred_points)[0]
    return np.percentile([d1, d2], 95)


In [2]:
import time
import torch

def measure_inference(model, img_path, central_point):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    
    start   = time.time()
    results = model(img_path, points=central_point, labels=[1], verbose=False)
    latency = (time.time() - start) * 1000  # Está en ms

    if torch.cuda.is_available():
        vram = torch.cuda.memory_allocated() / 1024**2 - vram_before
    else:
        vram = 0

    return results, latency, vram

def measure_inference_sam3_prompt(predictor, img_path, text_prompt):
    if torch.cuda.is_available():
        vram_before = torch.cuda.memory_allocated() / 1024**2
    start = time.time()
    predictor.set_image(img_path)
    predictor.model.set_classes(text=[text_prompt])
    predictor.prompts["text"] = [text_prompt]
    results = predictor()
    latency = (time.time() - start) * 1000
    vram = torch.cuda.memory_allocated() / 1024**2 - vram_before if torch.cuda.is_available() else 0
    return results, latency, vram

In [3]:
import numpy as np

def compute_all_metrics(pred_mask, gt_mask):
    """a"""
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
    tn = np.logical_and(~pred_mask, ~gt_mask).sum()

    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f2 = 5 * (precision * recall) / (4 * precision + recall) if (4 * precision + recall) > 0 else 0
    pixel_accuracy = (tp + tn) / (tp + fp + fn + tn) if (tp + fp + fn + tn) > 0 else 0
    return iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy

SAM 1 BASE

In [ ]:
import numpy as np
from ultralytics import SAM
import os
import cv2
import pandas as pd

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_b.pt")
    model.to("cuda")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores    = results[0].probs
            best_idx  = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":             ["sam_b"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
        df.to_csv(output_path, index=False)


evaluate_model()

SAM 1 GRANDE

In [ ]:
import numpy as np
from ultralytics import SAM
import os
import cv2
import pandas as pd

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam_l.pt")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores    = results[0].probs
            best_idx  = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":             ["sam_l"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
        df.to_csv(output_path, index=False)


evaluate_model()

loading annotations into memory...
Done (t=4.05s)
creating index...
index created!


## **SAM 2**

BASE

In [ ]:
import numpy as np
from ultralytics import SAM
import os
import cv2
import pandas as pd

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2_b.pt")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores    = results[0].probs
            best_idx  = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":             ["sam2_b"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
        df.to_csv(output_path, index=False)


evaluate_model()

loading annotations into memory...
Done (t=5.81s)
creating index...
index created!


GRANDE

In [ ]:
import numpy as np
from ultralytics import SAM
import os
import cv2
import pandas as pd

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2_l.pt")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores    = results[0].probs
            best_idx  = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":             ["sam2_l"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
        df.to_csv(output_path, index=False)


evaluate_model()

loading annotations into memory...
Done (t=6.23s)
creating index...
index created!


## **SAM 2.1**

2.1 BASE

In [ ]:
import numpy as np
from ultralytics import SAM
import os
import cv2
import pandas as pd

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2.1_b.pt")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores    = results[0].probs
            best_idx  = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":             ["sam2.1_b"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
        df.to_csv(output_path, index=False)


evaluate_model()

loading annotations into memory...
Done (t=4.86s)
creating index...
index created!


2.1 GRANDE

In [ ]:
import numpy as np
from ultralytics import SAM
import os
import cv2
import pandas as pd

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam2.1_l.pt")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores    = results[0].probs
            best_idx  = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":             ["sam2.1_l"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
        df.to_csv(output_path, index=False)


evaluate_model()

loading annotations into memory...
Done (t=4.41s)
creating index...
index created!


## **SAM 3**

In [ ]:
import numpy as np
from ultralytics import SAM
import os
import cv2
import pandas as pd

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

def get_bbox_from_mask(mask_binary):
    contours, _ = cv2.findContours(mask_binary.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    x, y, w, h = cv2.boundingRect(np.vstack(contours))
    return x, y, x + w, y + h


def evaluate_model():
    model = SAM("C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt")

    iou_scores         = []
    precision_scores   = []
    recall_scores      = []
    f1_scores          = []
    dice_scores        = []
    specificity_scores = []
    f2_scores          = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores     = []
    vram_scores        = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem   = mask_filename.replace("_Segmentation.png", "")
            img_path   = os.path.join(images_dir, img_stem + ".jpg")
            mask_path  = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)

            bbox = get_bbox_from_mask(gt_mask)
            if bbox is None:
                continue

            xmin, ymin, xmax, ymax = bbox
            central_point = [[(xmin + xmax) / 2, (ymin + ymax) / 2]]

            results, latency, vram = measure_inference(model, img_path, central_point)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            scores    = results[0].probs
            best_idx  = np.argmax(scores)
            pred_mask = results[0].masks.data[best_idx].cpu().numpy().astype(bool)

            H, W      = gt_mask.shape
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":             ["sam3"],
        "mean_iou":           [np.mean(iou_scores)],
        "mean_f1":            [np.mean(f1_scores)],
        "mean_recall":        [np.mean(recall_scores)],
        "mean_precision":     [np.mean(precision_scores)],
        "mean_dice":          [np.mean(dice_scores)],
        "mean_specificity":   [np.mean(specificity_scores)],
        "mean_f2":            [np.mean(f2_scores)],
        "mean_pixel_accuracy":[np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":  [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":  [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":    [np.mean(latency_scores)],
        "mean_vram_mb":       [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode='a', header=False, index=False)
    else:
        df.to_csv(output_path, index=False)


evaluate_model()

c:\Users\DanielTalavera\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading annotations into memory...
Done (t=3.44s)
creating index...
index created!
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[1024] must be multiple of max stride 14, updating to [1036]
WARNING imgsz=[10

SAM 3 USANDO PROMPTS PARA SEGMENTAR

In [ ]:
import numpy as np
from ultralytics.models.sam import SAM3SemanticPredictor
import cv2
import pandas as pd
import os
import torch
import time

DATASET_ROOT = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\ISIC_2016"

SPLITS = [
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Training_GroundTruth"),
    },
    {
        "images": os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_Data"),
        "masks":  os.path.join(DATASET_ROOT, "ISBI2016_ISIC_Part1_Test_GroundTruth"),
    },
]

TEXT_PROMPT = "skin lesion"

def evaluate_model():
    overrides = dict(
        conf=0.25,
        task="segment",
        mode="predict",
        model="C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\pesos_ultralytics\\sam3.pt",
        device="cuda",
    )
    predictor = SAM3SemanticPredictor(overrides=overrides)

    iou_scores            = []
    precision_scores      = []
    recall_scores         = []
    f1_scores             = []
    dice_scores           = []
    specificity_scores    = []
    f2_scores             = []
    pixel_accuracy_scores = []
    boundary_iou_scores   = []
    hausdorff_95_scores   = []
    latency_scores        = []
    vram_scores           = []

    for split in SPLITS:
        images_dir = split["images"]
        masks_dir  = split["masks"]

        for mask_filename in sorted(os.listdir(masks_dir)):
            if not mask_filename.lower().endswith(".png"):
                continue

            img_stem  = mask_filename.replace("_Segmentation.png", "")
            img_path  = os.path.join(images_dir, img_stem + ".jpg")
            mask_path = os.path.join(masks_dir, mask_filename)

            if not os.path.exists(img_path):
                continue

            gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            gt_mask = np.squeeze(gt_mask)
            gt_mask = (gt_mask > 127).astype(bool)
            if gt_mask.sum() == 0:
                continue

            results, latency, vram = measure_inference_sam3_prompt(predictor, img_path, TEXT_PROMPT)

            if results[0].masks is None or len(results[0].masks.data) == 0:
                continue

            masks = results[0].masks.data.cpu().numpy()
            H, W  = gt_mask.shape

            if len(masks) > 1:
                ious = []
                for m in masks:
                    m_resized    = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
                    intersection = np.logical_and(m_resized, gt_mask).sum()
                    union        = np.logical_or(m_resized, gt_mask).sum()
                    ious.append(intersection / union if union > 0 else 0)
                best_idx = np.argmax(ious)
            else:
                best_idx = 0

            pred_mask = masks[best_idx]
            pred_mask = cv2.resize(pred_mask.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)

            iou, precision, recall, f1, dice, specificity, f2, pixel_accuracy = compute_all_metrics(pred_mask, gt_mask)
            iou_scores.append(iou)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)
            dice_scores.append(dice)
            specificity_scores.append(specificity)
            f2_scores.append(f2)
            pixel_accuracy_scores.append(pixel_accuracy)
            boundary_iou_scores.append(boundary_iou(pred_mask, gt_mask))
            hausdorff_95_scores.append(hausdorff_95(pred_mask, gt_mask))
            latency_scores.append(latency)
            vram_scores.append(vram)

    resultados = {
        "modelo":              ["sam3_prompt"],
        "mean_iou":            [np.mean(iou_scores)],
        "mean_f1":             [np.mean(f1_scores)],
        "mean_recall":         [np.mean(recall_scores)],
        "mean_precision":      [np.mean(precision_scores)],
        "mean_dice":           [np.mean(dice_scores)],
        "mean_specificity":    [np.mean(specificity_scores)],
        "mean_f2":             [np.mean(f2_scores)],
        "mean_pixel_accuracy": [np.mean(pixel_accuracy_scores)],
        "mean_boundary_iou":   [np.mean(boundary_iou_scores)],
        "mean_hausdorff_95":   [np.mean(hausdorff_95_scores)],
        "mean_latency_ms":     [np.mean(latency_scores)],
        "mean_vram_mb":        [np.mean(vram_scores)],
    }

    df = pd.DataFrame(resultados)
    output_path = "C:\\Users\\DanielTalavera\\Desktop\\Trabajo_Fin_de_Grado\\resultados_zero_shot\\resultados_sam_isic2016.csv"

    if os.path.exists(output_path):
        df.to_csv(output_path, mode="a", header=False, index=False)
    else:
        df.to_csv(output_path, index=False)

evaluate_model()

loading annotations into memory...
Done (t=4.12s)
creating index...
index created!
Ultralytics 8.4.26  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3090, 24575MiB)
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\refcocog\train2014\COCO_train2014_000000380440.jpg: 644x644 1 the man in yellow coat, 191.3ms
Speed: 3.3ms preprocess, 191.3ms inference, 8.5ms postprocess per image at shape (1, 3, 644, 644)
Results saved to C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\SAM-zero-shot\runs\segment\predict25
WARNING imgsz=[640] must be multiple of max stride 14, updating to [644]

image 1/1 C:\Users\DanielTalavera\Desktop\Trabajo_Fin_de_Grado\refcocog\train2014\COCO_train2014_000000419645.jpg: 644x644 1 There is red colored truck in between the other trucks, 92.4ms
Speed: 3.3ms preprocess, 92.4ms inference, 2.4ms postprocess per image at shape (1, 3, 644, 644)
Results saved to C:\Users\